In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GaussNewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.04551320895552635
epoch:  1, loss: 0.0426655039191246
epoch:  2, loss: 0.04175557941198349
epoch:  3, loss: 0.040239572525024414
epoch:  4, loss: 0.034992992877960205
epoch:  5, loss: 0.03294610232114792
epoch:  6, loss: 0.029144583269953728
epoch:  7, loss: 0.022042058408260345
epoch:  8, loss: 0.01895594596862793
epoch:  9, loss: 0.018227731809020042
epoch:  10, loss: 0.014639757573604584
epoch:  11, loss: 0.013076279312372208
epoch:  12, loss: 0.011290541850030422
epoch:  13, loss: 0.010186774656176567
epoch:  14, loss: 0.009205222129821777
epoch:  15, loss: 0.008889419957995415
epoch:  16, loss: 0.008128009736537933


KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9858552813529968
Test metrics:  R2 = 0.9422004222869873


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GaussNewtonLS(
    model=model,
    batch_size=100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.4790589511394501
epoch:  1, loss: 0.39522987604141235
epoch:  2, loss: 0.26369667053222656
epoch:  3, loss: 0.18032489717006683
epoch:  4, loss: 0.11113157123327255
epoch:  5, loss: 0.05183850973844528
epoch:  6, loss: 0.0489453487098217
epoch:  7, loss: 0.046507637947797775
epoch:  8, loss: 0.04428231716156006
epoch:  9, loss: 0.04201104864478111
epoch:  10, loss: 0.0402815043926239
epoch:  11, loss: 0.038750190287828445
epoch:  12, loss: 0.03704823553562164
epoch:  13, loss: 0.03579717129468918
epoch:  14, loss: 0.0343242883682251
epoch:  15, loss: 0.03304755687713623
epoch:  16, loss: 0.03202429786324501
epoch:  17, loss: 0.031065354123711586
epoch:  18, loss: 0.03002086654305458
epoch:  19, loss: 0.029030626639723778
epoch:  20, loss: 0.028142472729086876
epoch:  21, loss: 0.02728678099811077
epoch:  22, loss: 0.026396948844194412
epoch:  23, loss: 0.025680184364318848
epoch:  24, loss: 0.02499276027083397
epoch:  25, loss: 0.024290109053254128
epoch:  26, loss: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8788729906082153
Test metrics:  R2 = 0.6435182094573975
